# Retrieval-Augmented Generation in One Notebook

Learning objective: understand how retrieval works, why it sometimes fails, and how better retrieval can support better answers.

## Before you start

If the libraries are not installed yet:

1. open a terminal
2. run:

```bash
python -m pip install -r requirements.txt
```

3. return to this notebook
4. restart the kernel if needed
5. run the cells from top to bottom

## What this notebook teaches

This notebook is one long teaching notebook.

We will move in a slow order:

1. prepare documents
2. split them into chunks
3. create embeddings
4. retrieve chunks for a question
5. inspect a failure case
6. improve retrieval with query rewrite
7. improve retrieval with HyDE
8. compare chunk sizes
9. build a context
10. inspect an evidence-based answer
11. optionally generate an answer with a local model

The goal is not just to get an answer.
The goal is to see what happens at each step.

## Local model note

Optional local generation in this notebook uses this default DataHub model path:

```text
/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf
```

If the file does not exist, the retrieval part of the notebook will still work.
Only the local generation step will be skipped.

In [ ]:
# numpy helps us work with numbers
import numpy

# sentence_transformers provides embedding models
from sentence_transformers import SentenceTransformer

# cosine_similarity compares vectors
from sklearn.metrics.pairwise import cosine_similarity

# os checks whether files exist
import os

# llama_cpp loads a local GGUF language model if available
try:
    from llama_cpp import Llama
    local_llm_support = True
except Exception:
    print("llama-cpp-python is not installed. The optional local generation step will be skipped.")
    local_llm_support = False

## Step 1: Prepare example documents

A **document collection** is the set of texts we want to search.

For teaching, we use short university documents.
That makes the intermediate steps easier to inspect by eye.

In [ ]:
documents = []

documents.append("The university library is open from 8 AM to 10 PM on weekdays. On weekends, the library is open from 10 AM to 6 PM.")
documents.append("Students can apply for financial aid between March 1 and April 15. The application requires academic records and financial documents.")
documents.append("Graduate students may apply for teaching assistant positions. Applications are reviewed by the department each semester.")
documents.append("The campus gym is open to students with a valid student ID. Guests need a temporary pass.")
documents.append("Parking permits for students must be renewed every semester. Permits can be purchased through the campus transportation office.")
documents.append("Scholarships are available for students with strong academic performance. Some scholarships require separate applications.")

## Inspect the raw documents

A retrieval system can only retrieve what is stored.

That means the first question is always:
**What text did we actually give the system?**

In [ ]:
document_index = 0

for document_text in documents:
    print("Document number:", document_index)
    print(document_text)
    print()

    document_index = document_index + 1

## Step 2: Split documents into chunks

A **chunk** is a smaller piece of text.

Why do we do this?

Because one document can contain many ideas.
If we search the whole document at once, retrieval can be less precise.
Chunks give retrieval smaller pieces to compare.

In [ ]:
chunk_size = 120
chunk_overlap = 20

chunk_records = []

document_index = 0

for document_text in documents:
    start_position = 0
    chunk_number = 0
    text_length = len(document_text)

    while start_position < text_length:
        end_position = start_position + chunk_size
        chunk_text = document_text[start_position:end_position]

        if chunk_text.strip() != "":
            chunk_record = {
                "source_name": "document_" + str(document_index),
                "chunk_number": chunk_number,
                "text": chunk_text.strip()
            }

            chunk_records.append(chunk_record)
            chunk_number = chunk_number + 1

        step_size = chunk_size - chunk_overlap

        if step_size <= 0:
            step_size = chunk_size

        start_position = start_position + step_size

    document_index = document_index + 1

## Inspect the chunks

This is one of the most important views in RAG.

If chunking is poor, retrieval is often poor.

In [ ]:
chunk_index = 0

for chunk_record in chunk_records:
    print("Chunk index:", chunk_index)
    print("Source:", chunk_record["source_name"])
    print("Chunk number:", chunk_record["chunk_number"])
    print(chunk_record["text"])
    print()

    chunk_index = chunk_index + 1

## Think about this

Look at the chunks.

Questions:
- Are the chunks too short?
- Are the chunks too long?
- Does each chunk contain one main idea, or several ideas mixed together?

In [ ]:
print("Current chunk size:")
print(chunk_size)

print()
print("Current chunk overlap:")
print(chunk_overlap)

## Step 3: Load an embedding model

An **embedding** is a vector of numbers that represents meaning.

The numbers themselves do not matter to humans.
What matters is that texts with similar meanings often get similar vectors.

In [ ]:
embedding_model_name = "all-MiniLM-L6-v2"

embedding_model = SentenceTransformer(embedding_model_name)

print("Loaded embedding model:")
print(embedding_model_name)

## Step 4: Create chunk embeddings

Now we convert each chunk into an embedding vector.

In [ ]:
chunk_embeddings = []

for chunk_record in chunk_records:
    chunk_embedding = embedding_model.encode(chunk_record["text"])
    chunk_embeddings.append(chunk_embedding)

print("Number of chunk embeddings:")
print(len(chunk_embeddings))

## Step 5: Ask a clear question

Before retrieval, make a prediction.

Which chunk do you think should rank near the top?

In [ ]:
clear_question = "When can students apply for financial aid?"

print("Clear question:")
print(clear_question)

clear_question_embedding = embedding_model.encode(clear_question)

## Compute similarity scores for the clear question

We use **cosine similarity** to compare the question vector with each chunk vector.

A higher score usually means a stronger semantic match.

In [ ]:
clear_similarity_scores = []

for chunk_embedding in chunk_embeddings:
    similarity_value = cosine_similarity([clear_question_embedding], [chunk_embedding])[0][0]
    clear_similarity_scores.append(similarity_value)

## Inspect all retrieval scores for the clear question

Do not skip this step.

A good RAG workflow includes inspecting retrieval.
We do not want retrieval to feel like a black box.

In [ ]:
chunk_index = 0

for similarity_value in clear_similarity_scores:
    print("Chunk index:", chunk_index)
    print("Similarity score:", similarity_value)
    print(chunk_records[chunk_index]["text"])
    print()

    chunk_index = chunk_index + 1

## Rank the chunks

Now we sort the chunks from best match to weakest match.

In [ ]:
clear_ranked_results = []

chunk_index = 0

for similarity_value in clear_similarity_scores:
    ranked_result = (
        similarity_value,
        chunk_records[chunk_index]["source_name"],
        chunk_records[chunk_index]["chunk_number"],
        chunk_records[chunk_index]["text"]
    )

    clear_ranked_results.append(ranked_result)
    chunk_index = chunk_index + 1

clear_ranked_results = sorted(clear_ranked_results, reverse=True)

## Inspect the top 3 retrieved chunks

A **top-k result** means the top few retrieved results.

Here, we will use k = 3.

In [ ]:
top_count = 0

for ranked_result in clear_ranked_results:
    if top_count < 3:
        print("Rank:", top_count + 1)
        print("Score:", ranked_result[0])
        print("Source:", ranked_result[1])
        print("Chunk number:", ranked_result[2])
        print("Chunk text:", ranked_result[3])
        print()

    top_count = top_count + 1

## What just happened?

The system compared the question with every chunk.
Then it ranked the chunks by similarity.

If the right chunk is near the top, retrieval worked well.

In [ ]:
top_result = clear_ranked_results[0]

print("Top retrieved score:")
print(top_result[0])

print()
print("Top retrieved text:")
print(top_result[3])

## Step 6: A failure case

Retrieval does not always work well.

A vague question often gives weaker retrieval results.

In [ ]:
vague_question = "When is aid?"

print("Vague question:")
print(vague_question)

vague_question_embedding = embedding_model.encode(vague_question)

## Compute similarity scores for the vague question

This repeats the same pattern on purpose.

That repetition helps us see exactly what changed.

In [ ]:
vague_similarity_scores = []

for chunk_embedding in chunk_embeddings:
    similarity_value = cosine_similarity([vague_question_embedding], [chunk_embedding])[0][0]
    vague_similarity_scores.append(similarity_value)

## Inspect retrieval results for the vague question

Think about these questions:

- Are the scores lower?
- Did the correct chunk move down?
- Why might the word "aid" be too weak or too vague?

In [ ]:
chunk_index = 0

for similarity_value in vague_similarity_scores:
    print("Chunk index:", chunk_index)
    print("Similarity score:", similarity_value)
    print(chunk_records[chunk_index]["text"])
    print()

    chunk_index = chunk_index + 1

## Step 7: Query rewrite

A **query rewrite** changes the user question into a clearer retrieval query.

This can improve retrieval because the rewritten question contains more information.

In [ ]:
rewritten_question = ""
rewritten_question = rewritten_question + "Provide detailed information about: "
rewritten_question = rewritten_question + vague_question
rewritten_question = rewritten_question + ". Include important dates, rules, and requirements if available."

print("Original question:")
print(vague_question)
print()
print("Rewritten question:")
print(rewritten_question)

rewritten_question_embedding = embedding_model.encode(rewritten_question)

## Retrieve again with the rewritten question

In [ ]:
rewritten_similarity_scores = []

for chunk_embedding in chunk_embeddings:
    similarity_value = cosine_similarity([rewritten_question_embedding], [chunk_embedding])[0][0]
    rewritten_similarity_scores.append(similarity_value)

## Inspect retrieval after query rewrite

Did the financial aid chunk move upward?
Did the score improve?

In [ ]:
chunk_index = 0

for similarity_value in rewritten_similarity_scores:
    print("Chunk index:", chunk_index)
    print("Similarity score:", similarity_value)
    print(chunk_records[chunk_index]["text"])
    print()

    chunk_index = chunk_index + 1

## Step 8: HyDE

HyDE means **Hypothetical Document Embedding**.

Instead of embedding the question directly, we first create a hypothetical document.
Then we embed that document and use it for retrieval.

This sometimes helps because the hypothetical document is richer than the short question.

In [ ]:
hypothetical_document = ""
hypothetical_document = hypothetical_document + "This document explains the topic: "
hypothetical_document = hypothetical_document + clear_question
hypothetical_document = hypothetical_document + ". It contains important facts, dates, requirements, and explanations."

print("Hypothetical document:")
print(hypothetical_document)

hypothetical_document_embedding = embedding_model.encode(hypothetical_document)

## Retrieve with HyDE

In [ ]:
hyde_similarity_scores = []

for chunk_embedding in chunk_embeddings:
    similarity_value = cosine_similarity([hypothetical_document_embedding], [chunk_embedding])[0][0]
    hyde_similarity_scores.append(similarity_value)

## Inspect retrieval for HyDE

Compare these results with:
- the clear question
- the vague question
- the rewritten question

In [ ]:
chunk_index = 0

for similarity_value in hyde_similarity_scores:
    print("Chunk index:", chunk_index)
    print("Similarity score:", similarity_value)
    print(chunk_records[chunk_index]["text"])
    print()

    chunk_index = chunk_index + 1

## Step 9: Compare chunk settings

Now we will run a mini chunking experiment.

We will use a smaller chunk size and compare the results.

In [ ]:
small_chunk_size = 80
small_chunk_overlap = 10

small_chunk_records = []

document_index = 0

for document_text in documents:
    start_position = 0
    chunk_number = 0
    text_length = len(document_text)

    while start_position < text_length:
        end_position = start_position + small_chunk_size
        chunk_text = document_text[start_position:end_position]

        if chunk_text.strip() != "":
            chunk_record = {
                "source_name": "document_" + str(document_index),
                "chunk_number": chunk_number,
                "text": chunk_text.strip()
            }

            small_chunk_records.append(chunk_record)
            chunk_number = chunk_number + 1

        step_size = small_chunk_size - small_chunk_overlap

        if step_size <= 0:
            step_size = small_chunk_size

        start_position = start_position + step_size

    document_index = document_index + 1

## Create embeddings for the smaller chunks

In [ ]:
small_chunk_embeddings = []

for chunk_record in small_chunk_records:
    chunk_embedding = embedding_model.encode(chunk_record["text"])
    small_chunk_embeddings.append(chunk_embedding)

## Retrieve with the smaller chunks

In [ ]:
small_chunk_clear_scores = []

for chunk_embedding in small_chunk_embeddings:
    similarity_value = cosine_similarity([clear_question_embedding], [chunk_embedding])[0][0]
    small_chunk_clear_scores.append(similarity_value)

## Inspect the top small-chunk results

Now compare these results with the earlier large-chunk results.

Questions:
- Did the top chunk change?
- Did the score change?
- Did a smaller chunk isolate the relevant sentence more clearly?

In [ ]:
small_chunk_ranked_results = []

chunk_index = 0

for similarity_value in small_chunk_clear_scores:
    ranked_result = (
        similarity_value,
        small_chunk_records[chunk_index]["source_name"],
        small_chunk_records[chunk_index]["chunk_number"],
        small_chunk_records[chunk_index]["text"]
    )

    small_chunk_ranked_results.append(ranked_result)
    chunk_index = chunk_index + 1

small_chunk_ranked_results = sorted(small_chunk_ranked_results, reverse=True)

top_count = 0

for ranked_result in small_chunk_ranked_results:
    if top_count < 3:
        print("Rank:", top_count + 1)
        print("Score:", ranked_result[0])
        print("Source:", ranked_result[1])
        print("Chunk number:", ranked_result[2])
        print("Chunk text:", ranked_result[3])
        print()

    top_count = top_count + 1

## Step 10: Build a context from top results

A **context** is the text we pass to the language model.

Here we build a context from the top retrieved chunks.

In [ ]:
context_text = ""

top_count = 0

for ranked_result in clear_ranked_results:
    if top_count < 3:
        context_text = context_text + ranked_result[3]
        context_text = context_text + "\n\n"

    top_count = top_count + 1

print("Context:")
print(context_text)

## Step 11: Build an evidence-based answer

Before calling a language model, it is useful to inspect the evidence directly.

This helps us ask:
- Is the retrieved evidence actually good?
- Would a generated answer be trustworthy?

In [ ]:
evidence_based_answer = ""
evidence_based_answer = evidence_based_answer + "Question: "
evidence_based_answer = evidence_based_answer + clear_question
evidence_based_answer = evidence_based_answer + "\n\n"
evidence_based_answer = evidence_based_answer + "Most relevant evidence:\n"
evidence_based_answer = evidence_based_answer + clear_ranked_results[0][3]

print(evidence_based_answer)

## Optional step: local generation with a GGUF model

This step is optional.

It needs:
- `llama-cpp-python`
- a GGUF model file

The default path below is the DataHub path.

In [ ]:
model_file_path = "/home/jovyan/shared/qwen2-1_5b-instruct-q4_0.gguf"

if local_llm_support is True and os.path.exists(model_file_path) is True:
    local_model = Llama(
        model_path=model_file_path,
        n_ctx=2048,
        n_threads=4,
        verbose=False
    )

    generation_prompt = ""
    generation_prompt = generation_prompt + "Use the context to answer the question.\n\n"
    generation_prompt = generation_prompt + "Context:\n"
    generation_prompt = generation_prompt + context_text
    generation_prompt = generation_prompt + "\nQuestion:\n"
    generation_prompt = generation_prompt + clear_question
    generation_prompt = generation_prompt + "\n\nAnswer:\n"

    generated_output = local_model(
        generation_prompt,
        max_tokens=200,
        temperature=0.2
    )

    generated_answer = generated_output["choices"][0]["text"]

    print("Generated answer:")
    print(generated_answer)
else:
    print("Local generation was skipped.")
    print("Either llama-cpp-python is missing or the GGUF model file was not found.")

## Reflection questions

1. Which retrieval setting worked best?
2. Why did the vague question perform worse?
3. Did query rewrite improve the match?
4. Did HyDE change the top result?
5. How did chunk size affect retrieval?
6. Was the generated answer consistent with the evidence?

## Summary

In this notebook, you practiced:

- document preparation
- chunking
- embeddings
- retrieval
- failure analysis
- query rewrite
- HyDE
- chunk-size comparison
- context building
- evidence-based inspection
- optional local generation

Most important idea:

Better retrieval usually leads to better answers.